## Predict 5′ cap composition from biological fingerprint data

This notebook predicts the most likely 5′ cap composition of an unknown biological sample using precomputed training libraries.

**Key features:**
- Uses centralized model registry for reproducibility
- Automatic model variant selection based on sample type
- Pre-loaded training libraries (no generation required)
- Built-in validation and diagnostics

See `models/MODEL_SELECTION_GUIDE.md` for detailed guidance on choosing the right model variant.

### 0. Load libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from rnacappredictor.predict_cap import predict_cap
from rnacappredictor.model_registry import (
    recommend_variant,
    get_model_config,
    get_available_variants,
    build_model_paths,
    print_model_info
)

### 1. Configuration
#### Define your sample and select the model variant

In [ ]:
# ============================================================
# USER CONFIGURATION
# ============================================================

SAMPLE_BATCH_NAME = "FM219"

FINGERPRINT_PATHS = [
    "/media/gibran/Data/Data/FM219/no_sample_id/20251015_1646_MD-101425_FBD19838_4faf599b/fastq_pass/U1-1_test/fingerprints.csv",
    "/media/gibran/Data/Data/FM219/no_sample_id/20251015_1646_MD-101425_FBD19838_4faf599b/fastq_pass/U1-11/fingerprints.csv",
    "/media/gibran/Data/Data/FM219/no_sample_id/20251015_1646_MD-101425_FBD19838_4faf599b/fastq_pass/U1-138P/fingerprints.csv",
    "/media/gibran/Data/Data/FM219/no_sample_id/20251015_1646_MD-101425_FBD19838_4faf599b/fastq_pass/U1-148P/fingerprints.csv",
    "/media/gibran/Data/Data/FM219/no_sample_id/20251015_1646_MD-101425_FBD19838_4faf599b/fastq_pass/U6/fingerprints.csv",
    "/media/gibran/Data/Data/FM219/no_sample_id/20251015_1646_MD-101425_FBD19838_4faf599b/fastq_pass/U4/fingerprints.csv",
]

# ============================================================
# MODEL SELECTION
# ============================================================

# Option 1: Automatic recommendation based on sample type
SAMPLE_TYPE = "blind"  # or "controlled", "validation"
HAS_INSDEL = False      # Do your fingerprints include INS/DEL columns?
RECOMMENDED_VERSION, RECOMMENDED_VARIANT = recommend_variant(SAMPLE_TYPE, HAS_INSDEL)

# Option 2: Manual selection (uncomment to override recommendation)
# RECOMMENDED_VERSION = "1.0"
# RECOMMENDED_VARIANT = "exclude_zero_FALSE_no_insdel"

print(f"Sample type: {SAMPLE_TYPE}")
print(f"Has INSDEL: {HAS_INSDEL}")
print(f"\n✓ Recommended model: {RECOMMENDED_VERSION}/{RECOMMENDED_VARIANT}")
print(f"\nTo see all available variants, run:")
print(f"  get_available_variants('{RECOMMENDED_VERSION}')")

# ============================================================
# PREDICTION PARAMETERS
# ============================================================

PRINT_TOP_K = 15
OUTPUT_DIR = "../results/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

### 1.5 Model Information
#### Display detailed information about your selected model

In [ ]:
# Show detailed information about the selected model variant
print_model_info(RECOMMENDED_VERSION, RECOMMENDED_VARIANT)

### 2. Load precomputed training library

In [ ]:
# ============================================================
# LOAD PRECOMPUTED TRAINING MIXTURES
# ============================================================

# Get the full configuration for this variant
model_config = get_model_config(RECOMMENDED_VERSION, RECOMMENDED_VARIANT)

# Build model file paths
model_paths = build_model_paths(
    RECOMMENDED_VERSION,
    RECOMMENDED_VARIANT,
    models_dir="../models"
)

# Load training mixture library
print(f"Loading training mixtures from: {model_paths['mixes']}")
df_train_mixes = pd.read_parquet(model_paths["mixes"])

print("\n✓ Loaded training mixture library:")
print(f"  Shape: {df_train_mixes.shape}")
print(f"  Columns: {df_train_mixes.columns.tolist()}")
print(f"  RTs: {df_train_mixes['RT'].unique().tolist()}")
print(f"  Unique caps (partial list): {list(df_train_mixes['cap'].unique())[:3]}...")

### 3. Load and concatenate biological fingerprint files

In [ ]:
# ============================================================
# LOAD BIOLOGICAL FINGERPRINT FILES
# ============================================================

def load_fingerprint_files(fingerprint_paths):
    """Load and concatenate fingerprint CSV files."""
    dfs = []
    for i, path in enumerate(fingerprint_paths):
        try:
            temp = pd.read_csv(path)
            temp["source_file"] = path
            dfs.append(temp)
            print(f"✓ Loaded {i+1}/{len(fingerprint_paths)}: {path}")
        except Exception as e:
            print(f"✗ Error loading {path}: {e}")
    return pd.concat(dfs, ignore_index=True) if dfs else None

df_test = load_fingerprint_files(FINGERPRINT_PATHS)

if df_test is not None:
    print(f"\n✓ Loaded biological fingerprints:")
    print(f"  Total rows: {df_test.shape[0]}")
    print(f"  Columns: {df_test.columns.tolist()}")
    display(df_test.head())
else:
    print("✗ Failed to load fingerprint files")

### 4. Barcode-to-RT mapping

In [ ]:
# ============================================================
# BARCODE TO RT MAPPING
# ============================================================
# Map barcode/isoform combinations to reverse transcriptase used

barcode_isoform_to_rt = {
    (1, "U1-1"): "INDURO",
    (6, "U1-1"): "ProtoScript",
    (11, "U1-1"): "Marathon",
    (16, "U1-1"): "GoScript",
    (21, "U1-1"): "EpiScript",

    (4, "U1-11"): "INDURO",
    (9, "U1-11"): "ProtoScript",
    (14, "U1-11"): "Marathon",
    (19, "U1-11"): "GoScript",
    (24, "U1-11"): "EpiScript",

    (2, "U1-138P"): "INDURO",
    (7, "U1-138P"): "ProtoScript",
    (12, "U1-138P"): "Marathon",
    (17, "U1-138P"): "GoScript",
    (22, "U1-138P"): "EpiScript",

    (3, "U1-148P"): "INDURO",
    (8, "U1-148P"): "ProtoScript",
    (13, "U1-148P"): "Marathon",
    (18, "U1-148P"): "GoScript",
    (23, "U1-148P"): "EpiScript",

    (5, "U6"): "INDURO",
    (10, "U6"): "ProtoScript",
    (15, "U6"): "Marathon",
    (20, "U6"): "GoScript",
    (1, "U6"): "EpiScript",

    (5, "U4"): "INDURO",
    (10, "U4"): "ProtoScript",
    (15, "U4"): "Marathon",
    (20, "U4"): "GoScript",
    (1, "U4"): "EpiScript",
}

print(f"✓ Barcode/isoform to RT mapping loaded")
print(f"  Total mappings: {len(barcode_isoform_to_rt)}")

### 5. Prepare the test data

In [ ]:
# ============================================================
# PREPARE TEST DATA
# ============================================================

df_test = df_test.copy()

# Convert barcode from "barcode01" / "barcode1" to integer if needed
if "barcode" in df_test.columns:
    df_test["barcode"] = df_test["barcode"].apply(
        lambda x: int(str(x).replace("barcode", ""))
    )

# Add RT using barcode + isoform mapping
df_test["RT"] = df_test.apply(
    lambda row: barcode_isoform_to_rt[(row["barcode"], row["isoform"])],
    axis=1
)

# Unknown cap label required by create_features()
df_test["cap"] = "unknown"

# Create separate experiment label per isoform/RNA
# (otherwise all RNAs treated as one sample)
df_test["experiment"] = SAMPLE_BATCH_NAME + "_" + df_test["isoform"].astype(str)

print("✓ Prepared test data:")
print(f"  Shape: {df_test.shape}")
print(f"\nExperiments to predict:")
print(df_test["experiment"].value_counts())
print(f"\nRTs by experiment:")
display(df_test.groupby("experiment")["RT"].unique())
display(df_test.head())

### 6. Validate data

In [ ]:
# ============================================================
# VALIDATE REQUIRED COLUMNS
# ============================================================

INCLUDE_INSDEL = model_config["include_insdel"]

if INCLUDE_INSDEL:
    required_columns = [
        "RT",
        "A%_INSDEL",
        "C%_INSDEL",
        "G%_INSDEL",
        "T%_INSDEL",
        "INS%_INSDEL",
        "DEL%_INSDEL",
        "cap",
        "experiment",
    ]
else:
    required_columns = [
        "RT",
        "A%",
        "C%",
        "G%",
        "T%",
        "cap",
        "experiment",
    ]

missing_columns = [col for col in required_columns if col not in df_test.columns]

if missing_columns:
    print(f"✗ ERROR: Missing required columns: {missing_columns}")
    raise ValueError(f"Missing required columns: {missing_columns}")
else:
    print(f"✓ All {len(required_columns)} required columns present")
    for col in required_columns:
        print(f"  • {col}")

### 7. Check barcode/isoform mappings

In [ ]:
# ============================================================
# CHECK BARCODE/ISOFORM MAPPING
# ============================================================

observed_pairs = set(zip(df_test["barcode"], df_test["isoform"]))

missing_pairs = [
    pair for pair in observed_pairs
    if pair not in barcode_isoform_to_rt
]

if missing_pairs:
    print(f"✗ ERROR: Missing barcode/isoform mappings:")
    for pair in sorted(missing_pairs):
        print(f"  {pair}")
    raise ValueError(f"Missing mappings: {missing_pairs}")
else:
    print(f"✓ All {len(observed_pairs)} barcode/isoform pairs are mapped")
    for pair in sorted(observed_pairs):
        rt = barcode_isoform_to_rt[pair]
        print(f"  {pair} → {rt}")

### 8. Check RT compatibility

In [ ]:
# ============================================================
# CHECK RT COMPATIBILITY
# ============================================================

training_rts = set(df_train_mixes["RT"].unique())
test_rts = set(df_test["RT"].unique())

missing_in_test = training_rts - test_rts
unknown_in_test = test_rts - training_rts

print(f"RTs in training: {sorted(training_rts)}")
print(f"RTs in test: {sorted(test_rts)}")

if missing_in_test:
    print(f"\n⚠️  WARNING: These RTs are in training but missing from test:")
    for rt in sorted(missing_in_test):
        print(f"  • {rt}")

if unknown_in_test:
    print(f"\n⚠️  WARNING: These RTs are in test but missing from training:")
    for rt in sorted(unknown_in_test):
        print(f"  • {rt}")

if not (missing_in_test or unknown_in_test):
    print(f"\n✓ Perfect RT compatibility between training and test!")

### 9. Run predictions

In [ ]:
# ============================================================
# RUN PREDICTION
# ============================================================

print(f"\nRunning predictions with model: {RECOMMENDED_VERSION}/{RECOMMENDED_VARIANT}")
print(f"Include INSDEL: {INCLUDE_INSDEL}")
print(f"\n{'='*80}\n")

results = predict_cap(
    df_train_mixes,
    df_test,
    show_true_cap=False,
    include_insdel=INCLUDE_INSDEL,
    print_top_k=PRINT_TOP_K,
    save_model=False
)

print(f"\n{'='*80}\n")
print(f"✓ Predictions complete!")
print(f"\nResults summary:")
display(results)

### 10. Save results

In [ ]:
# ============================================================
# SAVE RESULTS
# ============================================================

output_file = os.path.join(
    OUTPUT_DIR,
    f"{SAMPLE_BATCH_NAME}_{RECOMMENDED_VARIANT}_results.csv"
)

results.to_csv(output_file, index=False)
print(f"✓ Results saved to: {output_file}")

# Also save model version/variant for reproducibility
metadata_file = os.path.join(
    OUTPUT_DIR,
    f"{SAMPLE_BATCH_NAME}_{RECOMMENDED_VARIANT}_metadata.txt"
)

with open(metadata_file, "w") as f:
    f.write(f"Sample Batch: {SAMPLE_BATCH_NAME}\n")
    f.write(f"Model Version: {RECOMMENDED_VERSION}\n")
    f.write(f"Model Variant: {RECOMMENDED_VARIANT}\n")
    f.write(f"Exclude Zero Caps: {model_config['exclude_zero_caps']}\n")
    f.write(f"Include INSDEL: {model_config['include_insdel']}\n")
    f.write(f"Use Case: {model_config['use_case']}\n")
    f.write(f"\nGeneration Stats:\n")
    for key, value in model_config['generation_stats'].items():
        f.write(f"  {key}: {value}\n")

print(f"✓ Metadata saved to: {metadata_file}")